# 🌒 EX0005A | The Examiner Must Look
**Sunday, 15 November 2026 · Public instrument qualification · NOT a sealed efficacy test**

One night. One question. One independent verdict. The live Responses API section is **OFF by default**. The local G0 checker can run without a credential or API charges. Before any Luna call, complete and freeze the human-first predictions, review model/budget limits, and respect the **20:00 America/Denver** hard stop.

Source of research authorization: [EX0005A README](https://github.com/xxx3r/1001_nights/blob/main/nights/2026/11/2026-11-15-ex0005/README.md), [protocol draft](https://github.com/xxx3r/1001_nights/blob/main/nights/2026/11/2026-11-15-ex0005/EX0005A_PROTOCOL_DRAFT.md).

**No real-world target actions. No sealed holdouts. No model-written hypothesis represented as human-independent.**

## 1. Dependencies and reproducibility
This notebook writes both executable scripts from its own cells. It does not download or modify the source repository.

In [ ]:
%pip -q install "openai>=2.0" numpy torch

In [ ]:
from pathlib import Path
Path('ex0005a_public.py').write_text('"""EX0005A public synthetic instrument and evaluator. No sealed cases.\n\nThe model sees only tool-returned values, not oracle, healthy or lesion metadata.\nRun `python ex0005a_public.py` for local checker qualification.\n"""\nfrom __future__ import annotations\nimport copy, hashlib, json\nfrom dataclasses import dataclass\nfrom typing import Any\nimport numpy as np\nimport torch\n\nSEEDS = {"p1_eval": 150511, "p2_eval": 150527, "p1_probe": 150503}\nDIAG_LIMIT_PER_PATIENT = 4\nMAX_SCALARS_PER_PATCH = 6\nMAX_PATCH_ABS_DELTA = 2.0\n\n\ndef json_safe(obj: Any) -> Any:\n    if isinstance(obj, dict): return {str(k): json_safe(v) for k, v in obj.items()}\n    if isinstance(obj, (list, tuple)): return [json_safe(v) for v in obj]\n    if isinstance(obj, np.ndarray): return obj.tolist()\n    if isinstance(obj, np.generic): return obj.item()\n    return obj\n\n\ndef canonical_hash(payload: Any) -> str:\n    return hashlib.sha256(json.dumps(json_safe(payload), sort_keys=True, separators=(\',\', \':\')).encode()).hexdigest()\n\n\ndef params() -> dict[str, np.ndarray]:\n    return {"W1": np.eye(2, dtype=float), "b1": np.zeros(2),\n            "W2": np.array([1.0, 0.5]), "b2": np.array([0.1])}\n\n\ndef predict_torch(w: dict, x: np.ndarray) -> np.ndarray:\n    with torch.no_grad():\n        t = torch.tensor(x, dtype=torch.float64)\n        h = torch.relu(t @ torch.tensor(w["W1"], dtype=torch.float64).T + torch.tensor(w["b1"], dtype=torch.float64))\n        out = h @ torch.tensor(w["W2"], dtype=torch.float64) + torch.tensor(w["b2"], dtype=torch.float64)[0]\n        return out.numpy()\n\n\ndef predict_numpy(w: dict, x: np.ndarray) -> np.ndarray:\n    return np.maximum(x @ w["W1"].T + w["b1"], 0) @ w["W2"] + w["b2"][0]\n\n\ndef mse(w: dict, x: np.ndarray, y: np.ndarray) -> float:\n    return float(np.mean((predict_torch(w, x) - y)**2))\n\n\ndef rows(seed: int, n: int = 80) -> np.ndarray:\n    return np.random.default_rng(seed).uniform(0.2, 1.5, size=(n, 2))\n\n\ndef and_predict(weights: tuple[float, float, float, float, float]) -> np.ndarray:\n    a, b, bias, out_w, out_b = weights\n    x = np.array([[0,0], [0,1], [1,0], [1,1]], dtype=float)\n    return np.maximum(0, x @ np.array([a,b]) + bias)*out_w + out_b\n\n\ndef score_and(weights: tuple[float, ...], tol: float = 1e-8) -> dict:\n    assert len(weights) == 5 and all(np.isfinite(weights))\n    got = and_predict(weights)\n    target = np.array([0,0,0,1], dtype=float)\n    return {"outputs": got.tolist(), "max_abs_error": float(np.max(np.abs(got-target))),\n            "pass": bool(np.allclose(got, target, atol=tol, rtol=0))}\n\n\n@dataclass\nclass Patient:\n    name: str\n    w: dict\n    healthy: dict\n    x_eval: np.ndarray\n    y_eval: np.ndarray\n    x_probe: np.ndarray\n    y_probe: np.ndarray\n\n\ndef make_patient(name: str) -> Patient:\n    if name not in ("P1_plus", "P1_minus", "P2"):\n        raise ValueError("unknown public case")\n    healthy = params()\n    w = copy.deepcopy(healthy)\n    if name == "P1_plus": w["W2"][0] += .75\n    elif name == "P1_minus": w["W2"][0] -= .75\n    elif name == "P2": w["W1"][0, 0] -= .60\n    seed = SEEDS["p1_eval"] if name.startswith("P1") else SEEDS["p2_eval"]\n    x_eval = rows(seed)\n    # Same input and target data for the two causal-contrast variants.\n    x_probe = np.array([[.5,0.], [1.,0.], [1.5,0.], [0.,.5], [0.,1.], [1.,1.]], dtype=float)\n    return Patient(name,w,healthy,x_eval,predict_numpy(healthy,x_eval),x_probe,predict_numpy(healthy,x_probe))\n\n\ndef probe_examples(p: Patient, regime: str, indices: list[int]) -> dict:\n    regimes = {"feature0": [0,1,2], "feature1": [3,4], "mixed": [5]}\n    if regime not in regimes or not isinstance(indices,list) or not indices or len(indices)>3:\n        raise ValueError("invalid regime or indices")\n    if any(type(i) is not int or i not in regimes[regime] for i in indices):\n        raise ValueError("out of range for regime")\n    values = predict_torch(p.w,p.x_probe[indices]); targets=p.y_probe[indices]\n    return {"regime": regime, "rows": [{"index":i,"x":p.x_probe[i].tolist(),"target":float(t),"prediction":float(v),"residual":float(v-t)} for i,v,t in zip(indices,values,targets)],"provenance":"public development probe; not sealed"}\n\n\ndef inspect_weights(p: Patient, tensor: str, start: int, count: int) -> dict:\n    if tensor not in ("W1","b1","W2","b2") or type(start) is not int or type(count) is not int or count<=0 or count>4:\n        raise ValueError("invalid tensor/start/count")\n    a = p.w[tensor]; flat=a.ravel()\n    if start < 0 or start+count>len(flat): raise ValueError("tensor index out of bounds")\n    return {"tensor":tensor,"shape":list(a.shape),"flat_start":start,"values":flat[start:start+count].tolist(),"provenance":"current corrupted checkpoint"}\n\n\ndef apply_patch(p: Patient, patch: list[dict]) -> dict:\n    if not isinstance(patch,list) or not patch or len(patch)>MAX_SCALARS_PER_PATCH: raise ValueError("invalid patch edit count")\n    w=copy.deepcopy(p.w);seen=set()\n    for edit in patch:\n        if not isinstance(edit,dict) or set(edit)!={"tensor","index","delta"}: raise ValueError("patch requires tensor/index/delta")\n        t,i,d=edit["tensor"],edit["index"],edit["delta"]\n        if t not in w or type(i) is not int or type(d) not in (float,int) or not np.isfinite(d) or abs(d)>MAX_PATCH_ABS_DELTA: raise ValueError("invalid patch field")\n        if i<0 or i>=w[t].size or (t,i) in seen: raise ValueError("bad or duplicate index")\n        seen.add((t,i));w[t].flat[i]+=d\n    return w\n\n\ndef oracle_patch(p: Patient) -> list[dict]:\n    out=[]\n    for t in p.w:\n        for i,(a,b) in enumerate(zip(p.w[t].ravel(), p.healthy[t].ravel())):\n            if abs(a-b)>1e-12:out.append({"tensor":t,"index":i,"delta":float(b-a)})\n    return out\n\n\ndef heuristic_patch(p: Patient) -> list[dict]:\n    # Public, simple algebraic diagnostic baseline. Fix first-output feature response\n    # via output-layer coefficient; not designed to restore the internal mechanism.\n    r=probe_examples(p,"feature0",[1])["rows"][0]\n    slope_err=r["residual"] / r["x"][0]\n    return [{"tensor":"W2","index":0,"delta":-slope_err}]\n\n\ndef score_patch(p: Patient, patch: list[dict]) -> dict:\n    out=apply_patch(p,patch)\n    before=mse(p.w,p.x_eval,p.y_eval);after=mse(out,p.x_eval,p.y_eval)\n    return {"mse_before":before,"mse_after":after,"improvement":before-after,"patch":patch}\n\n\ndef prediction_delta(p: Patient, delta: float=-0.25) -> float:\n    trial=apply_patch(p,[{"tensor":"W2","index":0,"delta":delta}])\n    return mse(trial,p.x_eval,p.y_eval)-mse(p.w,p.x_eval,p.y_eval)\n\n\ndef qualification() -> dict:\n    # Independent analytical P0 solution, plus deliberately wrong all-zero negative.\n    p0=score_and((1,1,-1,1,0))\n    p0bad=score_and((0,0,0,0,0))\n    if not p0["pass"] or p0bad["pass"]: raise AssertionError("P0 checker flawed")\n    patients={n:make_patient(n) for n in ("P1_plus","P1_minus","P2")}\n    results={}\n    for name,p in patients.items():\n        x=p.x_eval\n        assert np.allclose(predict_torch(p.healthy,x),predict_numpy(p.healthy,x),atol=1e-12)\n        assert np.allclose(predict_torch(p.w,x),predict_numpy(p.w,x),atol=1e-12)\n        before=mse(p.w,x,p.y_eval)\n        oracle=score_patch(p,oracle_patch(p))\n        heuristic=score_patch(p,heuristic_patch(p))\n        # No-edit and direct-zero controls are deliberately included.\n        zero=[{"tensor":"W2","index":0,"delta":-float(p.w["W2"][0])}]\n        zero_score=score_patch(p,zero)\n        assert before>1e-4 and oracle["mse_after"]<1e-20\n        results[name]={"corrupt_mse":before,"oracle_mse":oracle["mse_after"],\n                       "no_edit_mse":before,"zero_mse":zero_score["mse_after"],\n                       "heuristic_mse":heuristic["mse_after"],\n                       "decrease_w2_0_0p25_delta_mse":prediction_delta(p)}\n    assert results["P1_plus"]["decrease_w2_0_0p25_delta_mse"] < 0\n    assert results["P1_minus"]["decrease_w2_0_0p25_delta_mse"] > 0\n    for malformed in [lambda:inspect_weights(patients["P1_plus"],"W2",2,1),\n                      lambda:probe_examples(patients["P1_plus"],"feature0",[4]),\n                      lambda:apply_patch(patients["P1_plus"],[{"tensor":"W2","index":0,"delta":5.}])]:\n        try: malformed(); raise AssertionError("malformed call accepted")\n        except ValueError: pass\n    # Deliberately incomplete/aborted response simulation cannot count as G1 success.\n    simulated_incomplete={"status":"incomplete","output":[]}\n    assert simulated_incomplete["status"]!="completed"\n    return {"status":"G0_LOCAL_PASS_ONLY","seeds":SEEDS,"p0_oracle":p0,"p0_bad":p0bad,\n            "cases":results,"paired_prediction": "delta_mse(P1_plus)<0; delta_mse(P1_minus)>0",\n            "malformed_requests_rejected":3,"simulated_truncation_rejected":True,\n            "public_fixture_fingerprint":canonical_hash({n:{t:a.tolist() for t,a in p.w.items()} for n,p in patients.items()}),\n            "live_model_tests":"NOT RUN"}\n\n\nif __name__=="__main__":\n    print(json.dumps(qualification(),indent=2,sort_keys=True))\n', encoding="utf-8")
print("Wrote ex0005a_public.py")

In [ ]:
from pathlib import Path
Path('run_luna.py').write_text('"""EX0005A Responses API live runner. Public data only; no credential logging.\n\nRequires human_fields.json completed BEFORE launch. Designed for Colab runtime.\nCosts API tokens. Source and human fields are hashed before model contact.\nNever run automatically on import.\n"""\nfrom __future__ import annotations\nimport argparse, datetime as dt, hashlib, json, os, pathlib, re, sys, traceback\nfrom zoneinfo import ZoneInfo\nimport numpy as np\nfrom ex0005a_public import (make_patient, qualification, inspect_weights, probe_examples,\n                           score_and, score_patch, prediction_delta, oracle_patch, heuristic_patch,\n                           canonical_hash, DIAG_LIMIT_PER_PATIENT)\n\nMODEL = \'gpt-6-luna\'\nMAX_RESPONSES = 16\nMAX_TOTAL_INPUT_TOKENS = 120000\nMAX_TOTAL_OUTPUT_TOKENS = 24000\nMAX_OUTPUT_TOKENS = 1800\nREASONING = {\'effort\': \'low\'}\nPREDICTION_TOL = 1e-9\nRESULT_DIR = pathlib.Path(\'ex0005a_receipts\')\nHUMAN_KEYS = (\'human_predicted_failure\',\'human_predicted_contrast\',\n              \'human_false_positive_test\',\'human_unsupported_mechanism\')\nTOOLS = [\n    {\'type\':\'function\',\'name\':\'inspect_weights\',\'description\':\'Read a bounded flat interval of CURRENT network tensor values. No healthy weights.\',\n     \'strict\':True,\'parameters\':{\'type\':\'object\',\'properties\':{\n         \'tensor\':{\'type\':\'string\',\'enum\':[\'W1\',\'b1\',\'W2\',\'b2\']},\n         \'start\':{\'type\':\'integer\'},\'count\':{\'type\':\'integer\'}},\n         \'required\':[\'tensor\',\'start\',\'count\'],\'additionalProperties\':False}},\n    {\'type\':\'function\',\'name\':\'probe_examples\',\'description\':\'Measure current model predictions, public target values and residuals on tiny fixed development examples.\',\n     \'strict\':True,\'parameters\':{\'type\':\'object\',\'properties\':{\n         \'regime\':{\'type\':\'string\',\'enum\':[\'feature0\',\'feature1\',\'mixed\']},\n         \'indices\':{\'type\':\'array\',\'items\':{\'type\':\'integer\'}}},\n         \'required\':[\'regime\',\'indices\'],\'additionalProperties\':False}}\n]\nBASE_INSTRUCTIONS = \'\'\'You are investigating ONE public synthetic 2-input, 2-ReLU-hidden, 1-linear-output network. Input features x0,x1 are nonnegative; W1 is shape (2,2), b1 shape (2,), W2 shape (2,), b2 shape (1,). There is one planted numerical corruption, but you do not know its location or sign. You have NO access to the healthy weights or private evaluator. Use only numerical tool evidence. You must take at least one valid observation using EACH of the two tools before diagnosis. All calls are read-only. Do not ask for a patch trial. Commit once, and do not infer mechanisms from successful function scores alone.\'\'\'\nOBS_PROMPT=\'\'\'Investigate the present corrupted network. Invoke inspect_weights and probe_examples with specific arguments; use numeric evidence. The final commitment must predict the signed change in mean squared error on an independent public development sample if W2\'s flat element 0 were DECREASED by exactly 0.25. No patch feedback will be provided before commitment. State your uncertain fault region and optional sparse patch. Do not guess from any formatting example.\'\'\'\nCOMMIT_PROMPT=\'\'\'Commit your diagnosis now, based only on already returned observations, without further tools. Return JSON ONLY, with exactly these keys: fault_region (e.g. an existing tensor and one flat element in square brackets), uncertainty (low/medium/high), predicted_delta_mse (numeric MSE_after_minus_before when W2[0] is decreased by 0.25), patch (array of at most six {tensor,index,delta} scalar edits, or empty array), explanation (at most 50 words). No extra keys, no code fences. An empty patch represents abstention.\'\'\'\n\n\ndef json_text(resp):\n    s=getattr(resp,\'output_text\',\'\') or \'\'\n    try: return json.loads(s)\n    except Exception: raise ValueError(\'model output is not strict JSON\')\n\n\ndef validate_commit(obj):\n    if not isinstance(obj,dict) or set(obj)!={\'fault_region\',\'uncertainty\',\'predicted_delta_mse\',\'patch\',\'explanation\'}: raise ValueError(\'malformed commitment keys\')\n    fr=obj[\'fault_region\']\n    m=re.fullmatch(r\'(W1|b1|W2|b2)\\[(\\d+)\\]\',fr) if isinstance(fr,str) else None\n    lengths={\'W1\':4,\'b1\':2,\'W2\':2,\'b2\':1}\n    if not m or int(m.group(2))>=lengths[m.group(1)]: raise ValueError(\'invalid fault region\')\n    if obj[\'uncertainty\'] not in (\'low\',\'medium\',\'high\'):raise ValueError(\'invalid uncertainty\')\n    val=obj[\'predicted_delta_mse\']\n    if type(val) not in (float,int) or not np.isfinite(val):raise ValueError(\'invalid numeric prediction\')\n    if not isinstance(obj[\'explanation\'],str) or len(obj[\'explanation\'].split())>50:raise ValueError(\'invalid explanation\')\n    patch=obj[\'patch\']\n    if not isinstance(patch,list) or len(patch)>6: raise ValueError(\'invalid patch\')\n    return obj\n\n\ndef safe_response(client, journal, label, **kwargs):\n    if sum(row.get(\'kind\')==\'api_response\' for row in journal) >= MAX_RESPONSES:\n        raise RuntimeError(\'hard cap on model generations reached\')\n    used_in=sum((row.get(\'usage\') or {}).get(\'input_tokens\',0) for row in journal if row.get(\'kind\')==\'api_response\')\n    used_out=sum((row.get(\'usage\') or {}).get(\'output_tokens\',0) for row in journal if row.get(\'kind\')==\'api_response\')\n    if used_in >= MAX_TOTAL_INPUT_TOKENS or used_out >= MAX_TOTAL_OUTPUT_TOKENS:\n        raise RuntimeError(\'hard cap on token usage reached\')\n    now=dt.datetime.now(ZoneInfo(\'America/Denver\'))\n    if now.date()!=dt.date(2026,11,15) or now.hour>=20:\n        raise RuntimeError(\'hard stop reached; no additional generation\')\n    resp=client.responses.create(model=MODEL,reasoning=REASONING,\n                                 max_output_tokens=MAX_OUTPUT_TOKENS,**kwargs)\n    usage=getattr(resp,\'usage\',None)\n    journal.append({\'kind\':\'api_response\',\'phase\':label,\n                    \'id\':getattr(resp,\'id\',None),\'model\':getattr(resp,\'model\',None),\n                    \'status\':getattr(resp,\'status\',None),\'incomplete_details\':str(getattr(resp,\'incomplete_details\',None)),\n                    \'usage\':usage.model_dump() if hasattr(usage,\'model_dump\') else None,\n                    \'output_items\':[(getattr(item,\'type\',None),getattr(item,\'name\',None)) for item in resp.output]})\n    if getattr(resp,\'status\',None)!=\'completed\':\n        raise RuntimeError(\'incomplete/failed response; not a valid observation or commitment\')\n    return resp\n\n\ndef p0_run(client,journal):\n    r=safe_response(client,journal,\'P0\',\n        instructions=\'Return exactly one JSON object. No tools are available.\',\n        input=\'For a 2-input -> 1 ReLU -> 1 linear scalar network, construct exact Boolean AND outputs on x in {(0,0),(0,1),(1,0),(1,1)}. Return JSON {"weights":[w0,w1,b_hidden,w_out,b_out]}, numeric entries only.\')\n    try:\n        obj=json_text(r)\n        if not isinstance(obj,dict) or set(obj)!={\'weights\'} or not isinstance(obj[\'weights\'],list): raise ValueError(\'bad P0 result\')\n        w=obj[\'weights\']\n        if len(w)!=5 or any(type(v) not in (float,int) for v in w):raise ValueError(\'bad weight array\')\n        return {\'valid\':True,\'model_weights\':w,\'evaluation\':score_and(tuple(w))}\n    except Exception as e:return {\'valid\':False,\'error\':str(e)}\n\n\ndef tool_dispatch(p, call, journal, label):\n    name=getattr(call,\'name\',\'?\')\n    try:\n        args=json.loads(call.arguments)\n        if not isinstance(args,dict): raise ValueError(\'expected object args\')\n        if name==\'inspect_weights\':\n            if set(args)!={\'tensor\',\'start\',\'count\'}:raise ValueError(\'bad property set\')\n            result=inspect_weights(p,**args)\n        elif name==\'probe_examples\':\n            if set(args)!={\'regime\',\'indices\'}:raise ValueError(\'bad property set\')\n            result=probe_examples(p,**args)\n        else:raise ValueError(\'unknown function\')\n        ok=True\n    except Exception as e:\n        result={\'error\':type(e).__name__+\': \'+str(e)};ok=False\n    journal.append({\'kind\':\'diagnostic\',\'case\':label,\'function\':name,\'arguments\':getattr(call,\'arguments\',\'\'),\n                    \'valid\':ok,\'return\':result})\n    return {\'type\':\'function_call_output\',\'call_id\':call.call_id,\'output\':json.dumps(result,allow_nan=False)},ok\n\n\ndef investigation(client, p, journal, label):\n    state={\'attempts\':0,\'valid\':0,\'kinds\':[],\'errors\':[],\'abstain\':False,\'commit\':None,\n           \'prediction_oracle\':prediction_delta(p)}\n    last_id=None;next_input=OBS_PROMPT\n    for turn in range(DIAG_LIMIT_PER_PATIENT):\n        args={\'instructions\':BASE_INSTRUCTIONS,\'input\':next_input,\'tools\':TOOLS,\'tool_choice\':\'required\'}\n        if last_id:args[\'previous_response_id\']=last_id\n        response=safe_response(client,journal,f\'{label}_diagnostic_{turn+1}\',**args)\n        calls=[o for o in response.output if getattr(o,\'type\',None)==\'function_call\']\n        if not calls:\n            state[\'errors\'].append(\'completed generation without function call\')\n            return state\n        outputs=[]\n        for call in calls:\n            if state[\'attempts\'] >= DIAG_LIMIT_PER_PATIENT:\n                # All emitted calls require replies; reject over-budget requests explicitly.\n                outputs.append({\'type\':\'function_call_output\',\'call_id\':call.call_id,\'output\':json.dumps({\'error\':\'diagnostic budget exhausted\'})})\n                journal.append({\'kind\':\'diagnostic\',\'case\':label,\'function\':call.name,\'valid\':False,\'error\':\'over budget\'})\n                state[\'errors\'].append(\'over budget\')\n                continue\n            state[\'attempts\']+=1\n            output,ok=tool_dispatch(p,call,journal,label)\n            outputs.append(output)\n            if ok:\n                state[\'valid\']+=1\n                if call.name not in state[\'kinds\']:state[\'kinds\'].append(call.name)\n            else:state[\'errors\'].append(\'invalid \'+call.name)\n        last_id=response.id;next_input=outputs\n        if len(state[\'kinds\'])>=2:break\n    if len(state[\'kinds\'])<2:\n        state[\'errors\'].append(\'missing distinct valid tool kinds\')\n        return state\n    final=safe_response(client,journal,f\'{label}_commit\',\n            instructions=BASE_INSTRUCTIONS+\' \'+COMMIT_PROMPT,\n            input=next_input,previous_response_id=last_id,tools=TOOLS,tool_choice=\'none\')\n    try:\n        obj=validate_commit(json_text(final))\n        state[\'commit\']=obj\n        state[\'abstain\']=not bool(obj[\'patch\'])\n        state[\'prediction_error\']=float(obj[\'predicted_delta_mse\']-state[\'prediction_oracle\'])\n        state[\'prediction_sign_correct\']=bool(np.sign(obj[\'predicted_delta_mse\'])==np.sign(state[\'prediction_oracle\']))\n        state[\'mechanism_region_correct\']=(obj[\'fault_region\']=={\'P1_plus\':\'W2[0]\',\'P1_minus\':\'W2[0]\',\'P2\':\'W1[0]\'}[label])\n        if obj[\'patch\']:\n            # Score only after commitment is frozen; no feedback to investigator.\n            try:state[\'patch_score\']=score_patch(p,obj[\'patch\'])\n            except Exception as e:state[\'errors\'].append(\'invalid committed patch \'+str(e))\n    except Exception as e:state[\'errors\'].append(\'invalid commitment \'+str(e))\n    return state\n\n\ndef human_fields(path):\n    d=json.loads(pathlib.Path(path).read_text())\n    for k in HUMAN_KEYS:\n        v=d.get(k)\n        if not isinstance(v,str) or len(v.strip())<15 or \'TO FILL\' in v.upper() or \'PLACEHOLDER\' in v.upper():\n            raise ValueError(\'human-owned field absent: \'+k)\n    if d.get(\'assistance_disclosure\') not in [\'I\',\'H\',\'C\',\'D\']:\n        raise ValueError(\'assistance_disclosure required\')\n    return d\n\n\ndef api_key():\n    # Colab secret first; local environment also supported. Never print either.\n    key=os.environ.get(\'OPENAI_API_KEY\',\'\')\n    if not key:\n        try:\n            from google.colab import userdata\n            key=userdata.get(\'OPENAI_API_KEY\') or \'\'\n        except ImportError:pass\n    if not key:raise RuntimeError(\'OPENAI_API_KEY not available; no live call performed\')\n    return key\n\n\ndef run_live(cfg):\n    from openai import OpenAI\n    now=dt.datetime.now(ZoneInfo(\'America/Denver\'))\n    if now.date()!=dt.date(2026,11,15) or (now.hour,now.minute)>=(20,0):\n        raise RuntimeError(\'EX0005A live window closed. Preserve NOT RUN; do not extend this night.\')\n    result_dir=RESULT_DIR\n    result_dir.mkdir(exist_ok=True)\n    journal=[]\n    base=pathlib.Path(__file__).parent\n    source_hashes={n:hashlib.sha256((base/n).read_bytes()).hexdigest() for n in (\'ex0005a_public.py\',\'run_luna.py\')}\n    # Source, human hypotheses, budgets and oracle-verified fixture fingerprint sealed locally before model contact.\n    q=qualification()\n    import importlib.metadata, platform\n    versions={\'python\':platform.python_version(),\'numpy\':np.__version__,\n              \'torch\':importlib.metadata.version(\'torch\'),\'openai\':importlib.metadata.version(\'openai\')}\n    manifest={\'datetime_frozen\':now.isoformat(),\'model_requested\':MODEL,\'software_versions\':versions,\n              \'api_generation_seed\':None,\'total_token_caps\':{\'input\':MAX_TOTAL_INPUT_TOKENS,\'output\':MAX_TOTAL_OUTPUT_TOKENS},\'settings\':{\'reasoning\':REASONING,\'max_output_tokens\':MAX_OUTPUT_TOKENS},\n              \'response_call_cap\':MAX_RESPONSES,\'diagnostic_limit_per_case\':DIAG_LIMIT_PER_PATIENT,\n              \'cases\':[\'P1_plus\',\'P1_minus\',\'P2\'],\'no_sealed_test\':True,\'precommitted_G0_fixture\':q[\'public_fixture_fingerprint\'],\n              \'source_sha256\':source_hashes,\'human_fields\':cfg,\'assistance\':cfg[\'assistance_disclosure\']}\n    manifest[\'freeze_sha256\']=canonical_hash(manifest)\n    (result_dir/\'PRE_LIVE_FREEZE.json\').write_text(json.dumps(manifest,indent=2,sort_keys=True))\n    client=OpenAI(api_key=api_key())\n    # This capability read makes no generation and reveals no credential; stop cleanly if not allowed.\n    summary={\'G0_local\':\'PASS\',\'model_requested\':MODEL,\'model_access_check\':\'NOT VERIFIED\',\n             \'freeze_sha256\':manifest[\'freeze_sha256\'],\'P0\':None,\'cases\':{},\'G1\':\'NOT RUN\',\'G2\':\'NOT RUN\',\'G3\':\'NOT RUN\',\'G4\':\'NOT RUN\',\'instrument_ready\':False}\n    try:\n        available=client.models.retrieve(MODEL)\n        summary[\'model_access_check\']=getattr(available,\'id\',None)\n        summary[\'P0\']=p0_run(client,journal)\n        for label in (\'P1_plus\',\'P1_minus\',\'P2\'):\n            # Every case is a fresh, separate model conversation.\n            p=make_patient(label)\n            summary[\'cases\'][label]=investigation(client,p,journal,label)\n        data=summary[\'cases\']\n        g1=all(len(data[k][\'kinds\'])>=2 and data[k][\'valid\']>=2 for k in (\'P1_plus\',\'P2\'))\n        g3=all(data[k][\'commit\'] is not None for k in (\'P1_plus\',\'P2\'))\n        # Matched-observation control: the requested diagnostic *types and exact arguments*\n        # must match across the sign-flipped pair. Otherwise, differing information or\n        # sampling histories confound attribution to the lesion-sign contrast.\n        def trace(label):\n            return [(j[\'function\'],j[\'arguments\']) for j in journal\n                    if j.get(\'kind\')==\'diagnostic\' and j.get(\'case\')==label and j.get(\'valid\')]\n        a,b=trace(\'P1_plus\'),trace(\'P1_minus\')\n        matched=bool(a) and a==b\n        relevant=any((name==\'probe_examples\' and json.loads(args)[\'regime\']==\'feature0\') or\n                     (name==\'inspect_weights\' and json.loads(args)[\'tensor\']==\'W2\' and\n                      json.loads(args)[\'start\']<=0<json.loads(args)[\'start\']+json.loads(args)[\'count\'])\n                     for name,args in a)\n        summary[\'paired_observation_contract\']={\'exact_call_signatures_matched\':matched,\n                                                \'relevant_evidence_requested\':relevant,\n                                                \'controlled_except_for_planted_weight\':True}\n        pair_commit=all(data[k][\'commit\'] is not None for k in (\'P1_plus\',\'P1_minus\'))\n        g2=pair_commit and all(data[k][\'prediction_sign_correct\'] for k in (\'P1_plus\',\'P1_minus\')) and (data[\'P1_plus\'][\'commit\'][\'predicted_delta_mse\'] < 0 < data[\'P1_minus\'][\'commit\'][\'predicted_delta_mse\'])\n        g2_status=(\'INCONCLUSIVE\' if not matched or not relevant else (\'PASS\' if g2 else \'FAIL\'))\n        summary.update({\'G1\':\'PASS\' if g1 else \'FAIL\',\'G2\':g2_status,\n                        \'G3\':\'PASS\' if g3 else \'FAIL\',\n                        \'G4\':\'SCORED\' if any(\'patch_score\' in d for d in data.values()) else \'NOT RUN\',\n                        \'instrument_ready\':bool(g1 and g2 and g3 and matched and relevant)})\n    except Exception as e:\n        summary[\'interruption\']={\'type\':type(e).__name__,\'message\':str(e)}\n        # No blanket retry. Incomplete results are still archived.\n    finally:\n        summary[\'public_baselines\']=q[\'cases\']\n        summary[\'api_responses_logged\']=sum(j[\'kind\']==\'api_response\' for j in journal)\n        (result_dir/\'ACTION_JOURNAL.json\').write_text(json.dumps(journal,indent=2,allow_nan=False))\n        (result_dir/\'RESULTS.json\').write_text(json.dumps(summary,indent=2,allow_nan=False))\n    return summary\n\n\ndef main():\n    ap=argparse.ArgumentParser()\n    ap.add_argument(\'--human-fields\',default=\'human_fields.json\')\n    ap.add_argument(\'--local-only\',action=\'store_true\',help=\'No API usage; run G0 only.\')\n    ap.add_argument(\'--live\',action=\'store_true\',help=\'Explicitly authorize bounded billable calls in live window.\')\n    args=ap.parse_args()\n    if args.local_only:\n        print(json.dumps(qualification(),indent=2));return\n    if not args.live:ap.error(\'Choose --local-only or --live; live never starts implicitly.\')\n    cfg=human_fields(args.human_fields)\n    summary=run_live(cfg)\n    print(json.dumps({\'freeze_sha256\':summary[\'freeze_sha256\'], \'G1\':summary[\'G1\'],\n                      \'G2\':summary[\'G2\'],\'G3\':summary[\'G3\'],\'G4\':summary[\'G4\'],\n                      \'instrument_ready\':summary[\'instrument_ready\'],\n                      \'receipts\':str(RESULT_DIR)},indent=2))\n\nif __name__==\'__main__\':main()\n', encoding="utf-8")
print("Wrote run_luna.py")

In [ ]:
from pathlib import Path
Path('test_offline.py').write_text('"""Independent small negative controls; no OpenAI calls."""\nimport json\nimport unittest\nimport numpy as np\nfrom ex0005a_public import *\nfrom run_luna import validate_commit, human_fields\n\nclass PublicChecker(unittest.TestCase):\n    def test_and_oracle_and_bad(self):\n        self.assertTrue(score_and((1,1,-1,1,0))[\'pass\'])\n        self.assertFalse(score_and((0,0,0,0,0))[\'pass\'])\n    def test_torch_numpy_agree(self):\n        for n in [\'P1_plus\',\'P1_minus\',\'P2\']:\n            p=make_patient(n)\n            np.testing.assert_allclose(predict_numpy(p.w,p.x_eval),predict_torch(p.w,p.x_eval),atol=1e-12)\n    def test_paired_is_matched_except_one_scalar(self):\n        a,b=make_patient(\'P1_plus\'),make_patient(\'P1_minus\')\n        np.testing.assert_array_equal(a.x_eval,b.x_eval)\n        np.testing.assert_array_equal(a.y_eval,b.y_eval)\n        for t in (\'W1\',\'b1\',\'b2\'):np.testing.assert_array_equal(a.w[t],b.w[t])\n        self.assertEqual(float(a.w[\'W2\'][1]),float(b.w[\'W2\'][1]))\n        self.assertAlmostEqual(float(a.w[\'W2\'][0]-b.w[\'W2\'][0]),1.5)\n        self.assertLess(prediction_delta(a),0)\n        self.assertGreater(prediction_delta(b),0)\n    def test_negative_control_scoring(self):\n        r=qualification()\n        self.assertEqual(r[\'status\'],\'G0_LOCAL_PASS_ONLY\')\n        for x in r[\'cases\'].values():\n            self.assertGreater(x[\'no_edit_mse\'],1e-4)\n            self.assertLess(x[\'oracle_mse\'],1e-15)\n            self.assertGreater(x[\'zero_mse\'],x[\'oracle_mse\'])\n    def test_malformed_tool_contract(self):\n        p=make_patient(\'P1_plus\')\n        for f in [lambda:inspect_weights(p,\'W2\',3,1),\n                  lambda:inspect_weights(p,\'W2\',0,50),\n                  lambda:probe_examples(p,\'feature0\',[3]),\n                  lambda:probe_examples(p,\'feature1\',[]),\n                  lambda:apply_patch(p,[{\'tensor\':\'W2\',\'index\':0,\'delta\':0.5},{\'tensor\':\'W2\',\'index\':0,\'delta\':0.3}]),\n                  lambda:apply_patch(p,[{\'tensor\':\'W2\',\'index\':0,\'delta\':float(\'nan\')}])]:\n            with self.assertRaises(ValueError): f()\n    def test_commit_checker(self):\n        correct={\'fault_region\':\'W2[0]\',\'uncertainty\':\'high\',\'predicted_delta_mse\':-.2,\'patch\':[],\'explanation\':\'A tentative numerical diagnosis.\'}\n        self.assertEqual(validate_commit(correct),correct)\n        for wrong in [{**correct,\'fault_region\':\'W2[99]\'},\n                      {**correct,\'predicted_delta_mse\':float(\'nan\')},\n                      {**correct,\'fault_region\':\'not-a-tensor\'}]:\n            with self.assertRaises(ValueError):validate_commit(wrong)\n    def test_human_gate_is_closed(self):\n        import tempfile\n        from pathlib import Path\n        with tempfile.TemporaryDirectory() as tmp:\n            path=Path(tmp)/\'human_fields.json\'\n            path.write_text(json.dumps({\n                \'human_predicted_failure\':\'[HUMAN TO FILL]\',\n                \'human_predicted_contrast\':\'[HUMAN TO FILL]\',\n                \'human_false_positive_test\':\'[HUMAN TO FILL]\',\n                \'human_unsupported_mechanism\':\'[HUMAN TO FILL]\',\n                \'assistance_disclosure\':\'D\'}))\n            with self.assertRaises(ValueError):human_fields(path)\n\nif __name__ == \'__main__\':unittest.main(verbosity=2)\n', encoding="utf-8")
print("Wrote test_offline.py")

## 2. Non-billable G0 checker and negative controls
This provides numeric fixture readiness only. A successful checker does **not** imply Luna has investigated or repaired anything.

In [ ]:
!python ex0005a_public.py
!python -m unittest -v test_offline

## 3. Human-first prediction, to be filled BEFORE live evaluation
Write in your own words. Assistance disclosure: **I** independent, **H** hint, **C** collaborative, **D** AI-drafted. The runner rejects unfilled placeholders. An AI contribution must be disclosed rather than passed off as human reasoning.

In [ ]:
import json
from pathlib import Path
human_fields = {
    "human_predicted_failure": "[HUMAN TO FILL before any live model generation]",
    "human_predicted_contrast": "[HUMAN TO FILL before any live model generation]",
    "human_false_positive_test": "[HUMAN TO FILL before any live model generation]",
    "human_unsupported_mechanism": "[HUMAN TO FILL before any live model generation]",
    "assistance_disclosure": "D",  # change if your actual assistance differs
}
Path('human_fields.json').write_text(json.dumps(human_fields, indent=2))
print('Human-first fields file prepared. Fill placeholders before a live run.')

## 4. Real GPT-6 Luna API preflight and bounded investigation (explicit opt-in)

Create `OPENAI_API_KEY` in **Colab Secrets** (left-hand key icon), and enable notebook access. The runner references the secret without printing/storing it. Official Responses API function-calling documentation: https://developers.openai.com/api/docs/guides/function-calling

**Frozen proposed limits:** `gpt-6-luna`; 3 public cases P1+, P1− and P2; two read-only tools; 4 diagnostic requests per case; 16 total generation calls; 1,800 maximum output tokens/call; 120,000 cumulative input-token cap, 24,000 output-token cap; low reasoning. No automatic network retries, no sealed evaluation, at most six scalar edits per patch. All model outputs and tool errors are journaled. The runner refuses model calls after 20:00 America/Denver on 15 November 2026.

Human fields and source hashes are frozen locally **before** first model access. **Do not enable this until you intend to spend API credits.**

In [ ]:
import subprocess
RUN_BILLABLE_API = False  # Explicitly change to True only after the pre-live checks
if RUN_BILLABLE_API:
    proc = subprocess.run(['python','run_luna.py','--live'], check=False)
    print('Live runner exit status:',proc.returncode)
else:
    print('Live API intentionally not run. G1/G2/G3/G4 remain NOT RUN.')

## 5. Inspect receipts and close the night
Read the three files if a real run occurred. Do not treat mock or G0 data as Luna results. If no valid model trial ran, close **NOT RUN** at the deadline; if a valid trial failed a frozen check, report **FAIL**. Other status labels remain available for genuinely inconclusive evidence.

In [ ]:
from pathlib import Path
p = Path('ex0005a_receipts')
if p.is_dir():
    for f in ['PRE_LIVE_FREEZE.json', 'RESULTS.json']:
        print('\n---', f, '---')
        print((p / f).read_text()[:12000] if (p/f).exists() else 'MISSING')
else:
    print('No live receipts. Model investigation NOT RUN. Local G0 is separate.')